# Sales Data Analysis

**Goal:** explore retail transactions, measure revenue and profit, identify product/region/channel patterns, and communicate findings.

> **Dataset note:** `sales_transactions.csv` is a synthetic dataset created for portfolio practice. It does not represent a real company or real customers.

## Business questions
1. How do revenue and profit change over time?
2. Which products and categories contribute most to revenue and profit?
3. Which regions and sales channels perform best?
4. How large is the effect of discounts on profit?

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("../data/sales_transactions.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["order_date"])
pd.set_option("display.max_columns", None)
df.head()

## 1. Initial inspection

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T)
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Unique order IDs:", df["order_id"].nunique())

## 2. Data validation and feature engineering

In [ ]:
# Check basic business rules before analysis
assert df["order_id"].is_unique, "Order IDs should be unique"
assert (df["quantity"] > 0).all(), "Quantity must be positive"
assert df["discount_rate"].between(0, 1).all(), "Discount must be between 0 and 1"
assert (df[["unit_price", "unit_cost", "revenue", "cost"]] >= 0).all().all()

# Recalculate financial fields to verify consistency
df["calculated_revenue"] = (df["unit_price"] * df["quantity"] * (1-df["discount_rate"])).round(2)
df["revenue_check"] = (df["revenue"] - df["calculated_revenue"]).abs() < 0.02
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["year"] = df["order_date"].dt.year
df["profit_margin"] = np.where(df["revenue"] != 0, df["profit"] / df["revenue"], np.nan)

print("Revenue calculation matches:", df["revenue_check"].mean())
display(df[["order_date","product","revenue","profit","profit_margin"]].head())

## 3. Key performance indicators (KPIs)

In [ ]:
total_revenue = df["revenue"].sum()
total_profit = df["profit"].sum()
total_orders = df["order_id"].nunique()
total_units = df["quantity"].sum()
average_order_value = total_revenue / total_orders
overall_margin = total_profit / total_revenue

kpis = pd.Series({
    "Total revenue (EUR)": total_revenue,
    "Total profit (EUR)": total_profit,
    "Orders": total_orders,
    "Units sold": total_units,
    "Average order value (EUR)": average_order_value,
    "Profit margin": overall_margin
})
display(kpis.to_frame("value").round(2))

## 4. Monthly performance

In [ ]:
monthly = df.groupby("month", as_index=False).agg(
    revenue=("revenue","sum"),
    profit=("profit","sum"),
    orders=("order_id","nunique")
)
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(monthly["month"], monthly["revenue"], marker="o", label="Revenue")
ax.plot(monthly["month"], monthly["profit"], marker="o", label="Profit")
ax.set_title("Monthly Revenue and Profit")
ax.set_xlabel("Month"); ax.set_ylabel("EUR")
ax.tick_params(axis="x", rotation=70); ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.show()
display(monthly.tail(10))

## 5. Product and category performance

In [ ]:
product_perf = df.groupby("product", as_index=False).agg(
    revenue=("revenue","sum"),
    profit=("profit","sum"),
    units=("quantity","sum")
).sort_values("revenue", ascending=False)

fig, ax = plt.subplots(figsize=(10,5))
ax.barh(product_perf["product"], product_perf["revenue"])
ax.invert_yaxis(); ax.set_title("Revenue by Product")
ax.set_xlabel("Revenue (EUR)")
plt.tight_layout(); plt.show()
display(product_perf.round(2))

category_perf = df.groupby("category", as_index=False).agg(
    revenue=("revenue","sum"), profit=("profit","sum")
).sort_values("revenue", ascending=False)
display(category_perf.round(2))

## 6. Region and channel performance

In [ ]:
region_perf = df.groupby("region", as_index=False).agg(
    revenue=("revenue","sum"), profit=("profit","sum"), orders=("order_id","nunique")
).sort_values("revenue", ascending=False)
channel_perf = df.groupby("sales_channel", as_index=False).agg(
    revenue=("revenue","sum"), profit=("profit","sum"), orders=("order_id","nunique")
).sort_values("revenue", ascending=False)

fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].bar(region_perf["region"], region_perf["revenue"])
axes[0].set_title("Revenue by Region"); axes[0].set_ylabel("EUR")
axes[1].bar(channel_perf["sales_channel"], channel_perf["revenue"])
axes[1].set_title("Revenue by Sales Channel")
plt.tight_layout(); plt.show()
display(region_perf.round(2)); display(channel_perf.round(2))

## 7. Discount analysis

In [ ]:
discount_perf = df.groupby("discount_rate", as_index=False).agg(
    revenue=("revenue","sum"),
    profit=("profit","sum"),
    avg_margin=("profit_margin","mean"),
    orders=("order_id","count")
).sort_values("discount_rate")
display(discount_perf.round(3))

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(discount_perf["discount_rate"]*100, discount_perf["avg_margin"], marker="o")
ax.set_title("Average Profit Margin by Discount Level")
ax.set_xlabel("Discount (%)"); ax.set_ylabel("Average profit margin")
ax.grid(alpha=.25); plt.tight_layout(); plt.show()

## 8. Automated summary and export

In [ ]:
best_product = product_perf.iloc[0]
best_region = region_perf.iloc[0]
best_channel = channel_perf.iloc[0]
summary = f'''# Sales Analysis Summary

- Total revenue: EUR {total_revenue:,.2f}
- Total profit: EUR {total_profit:,.2f}
- Number of orders: {total_orders:,}
- Units sold: {total_units:,}
- Average order value: EUR {average_order_value:,.2f}
- Overall profit margin: {overall_margin:.1%}
- Highest-revenue product: {best_product['product']} (EUR {best_product['revenue']:,.2f})
- Highest-revenue region: {best_region['region']} (EUR {best_region['revenue']:,.2f})
- Highest-revenue channel: {best_channel['sales_channel']} (EUR {best_channel['revenue']:,.2f})

**Interpretation:** These are descriptive findings from a synthetic practice dataset. They should not be treated as evidence about a real business.
'''
print(summary)
Path("../reports").mkdir(exist_ok=True)
Path("../reports/analysis_summary.md").write_text(summary, encoding="utf-8")
monthly.to_csv("../reports/monthly_performance.csv", index=False)
product_perf.to_csv("../reports/product_performance.csv", index=False)
print("Reports exported.")

## 9. Recommendations and limitations

Write 3–5 recommendations based on the patterns you observe. Do not assume that correlation proves causation. For example, a high-revenue product is not automatically the most profitable product.

**Portfolio customization checklist**
- Replace the synthetic dataset with a public dataset if desired.
- Add your own interpretation in the markdown cells.
- Include 2–3 screenshots of the strongest charts in the README.
- State clearly that this is an educational project and explain your contribution.
